# Wrist-motion data trimmer

Choose a recording, inspect its synchronized sensor channels, select a time range, and export a trimmed CSV. The original files are never modified.

1. Run all cells.
2. Select a CSV and sensor group.
3. Drag the **Time range** handles.
4. Choose whether to keep or remove the selected interval.
5. Click **Export trimmed CSV**.

In [6]:
from pathlib import Path
import math
import os
 
# Use a project-local cache so Matplotlib does not rebuild its font index.
MPL_CACHE = Path('.cache/matplotlib')
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CACHE.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

DATA_DIR = Path('/Users/aawni/Projects/model/data/logged_data')
OUTPUT_DIR = Path('/Users/aawni/Projects/model/data/trimmed_data')
MAX_PLOT_POINTS = 2_500

TIME_COLUMN = 'secondsElapsed'
CONTEXT_COLUMNS = ['weight', 'set', 'reps', 'activity', 'activityEncoded']
SENSOR_GROUPS = {
    'Rotation rate': [
        'wristMotion_rotationRateX', 'wristMotion_rotationRateY', 'wristMotion_rotationRateZ'
    ],
    'Gravity': [
        'wristMotion_gravityX', 'wristMotion_gravityY', 'wristMotion_gravityZ'
    ],
    'Acceleration': [
        'wristMotion_accelerationX', 'wristMotion_accelerationY', 'wristMotion_accelerationZ'
    ],
    'Quaternion': [
        'wristMotion_quaternionW', 'wristMotion_quaternionX',
        'wristMotion_quaternionY', 'wristMotion_quaternionZ'
    ],
}
REQUIRED_COLUMNS = [TIME_COLUMN, *sum(SENSOR_GROUPS.values(), [])]
COLORS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']

csv_files = sorted(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR.resolve()}')

plt.style.use('seaborn-v0_8-whitegrid')
print(f'Found {len(csv_files)} recordings in {DATA_DIR}.')

Found 15 recordings in /Users/aawni/Projects/model/data/logged_data.


In [ ]:
# Interactive controls and trimming logic
file_picker = widgets.Dropdown(
    options=[(path.name, path) for path in csv_files],
    description='Recording:',
    layout=widgets.Layout(width='720px'),
    style={'description_width': '90px'},
)
group_picker = widgets.Dropdown(
    options=['All sensor groups', *SENSOR_GROUPS],
    value='Rotation rate',
    description='Plots:',
    layout=widgets.Layout(width='310px'),
    style={'description_width': '90px'},
)
trim_mode = widgets.ToggleButtons(
    options=[('Keep selected range', 'keep'), ('Remove selected range', 'remove')],
    value='keep',
    description='Action:',
    style={'description_width': '90px'},
)
reset_time = widgets.Checkbox(
    value=False,
    description='Reset exported time to 0',
    indent=False,
)
load_button = widgets.Button(
    description='Load recording',
    button_style='primary',
    icon='folder-open',
)
time_range = widgets.FloatRangeSlider(
    value=(0.0, 1.0), min=0.0, max=1.0, step=0.01,
    description='Time range:',
    continuous_update=False,
    readout_format='.2f',
    layout=widgets.Layout(width='95%'),
    style={'description_width': '90px'},
    disabled=True,
)
export_button = widgets.Button(
    description='Export trimmed CSV',
    button_style='success',
    icon='save',
)
plot_output = widgets.Output()
status_output = widgets.Output()

current_df = None
current_path = None

def _first_value(frame, column):
    values = frame[column].dropna()
    return values.iloc[0] if len(values) else '—'

def _selection_mask(frame):
    start, end = time_range.value
    elapsed = pd.to_numeric(frame[TIME_COLUMN], errors='coerce')
    return elapsed.between(start, end, inclusive='both')

def _trimmed_frame():
    mask = _selection_mask(current_df)
    result = current_df.loc[mask if trim_mode.value == 'keep' else ~mask].copy()
    if reset_time.value and not result.empty:
        valid_time = pd.to_numeric(result[TIME_COLUMN], errors='coerce')
        first_time = valid_time.dropna().iloc[0] if valid_time.notna().any() else 0
        result[TIME_COLUMN] = valid_time - first_time
    return result

def load_recording(*_):
    global current_df, current_path
    current_path = Path(file_picker.value)
    frame = pd.read_csv(current_path, low_memory=False)
    missing = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError('Missing required columns: ' + ', '.join(missing))
    frame[TIME_COLUMN] = pd.to_numeric(frame[TIME_COLUMN], errors='coerce')
    current_df = frame

    valid_time = frame[TIME_COLUMN].dropna()
    if valid_time.empty:
        raise ValueError(f'{current_path.name} has no valid {TIME_COLUMN} values.')
    t_min, t_max = float(valid_time.min()), float(valid_time.max())
    diffs = valid_time.sort_values().diff().dropna()
    step = max(float(diffs.median()) if len(diffs) else 0.01, 0.001)
    time_range.min = min(time_range.min, t_min)
    time_range.max = max(time_range.max, t_max)
    time_range.min = t_min
    time_range.max = t_max
    time_range.step = step
    time_range.value = (t_min, t_max)
    time_range.disabled = False
    plot_selection()

def plot_selection(*_):
    if current_df is None:
        return
    frame = current_df
    start, end = time_range.value
    selected = _selection_mask(frame)
    result_rows = int(selected.sum() if trim_mode.value == 'keep' else (~selected).sum())
    groups = SENSOR_GROUPS if group_picker.value == 'All sensor groups' else {
        group_picker.value: SENSOR_GROUPS[group_picker.value]
    }
    stride = max(1, math.ceil(len(frame) / MAX_PLOT_POINTS))
    plot_frame = frame.iloc[::stride]

    with plot_output:
        clear_output(wait=True)
        display(HTML(
            f'<b>{current_path.name}</b><br>'
            f'Selected: <b>{int(selected.sum()):,}</b> of {len(frame):,} rows. '
            f'Export result: <b>{result_rows:,}</b> rows.'
        ))
        fig, axes = plt.subplots(
            len(groups), 1,
            figsize=(14, 3.0 * len(groups)),
            sharex=True,
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes)
        for axis, (group_name, columns) in zip(axes, groups.items()):
            for color, column in zip(COLORS, columns):
                label = column.rsplit('_', 1)[-1]
                axis.plot(
                    plot_frame[TIME_COLUMN],
                    pd.to_numeric(plot_frame[column], errors='coerce'),
                    color=color, linewidth=0.9, label=label, rasterized=True,
                )
            if trim_mode.value == 'keep':
                axis.axvspan(time_range.min, start, color='#D55E00', alpha=0.13)
                axis.axvspan(end, time_range.max, color='#D55E00', alpha=0.13)
            else:
                axis.axvspan(start, end, color='#D55E00', alpha=0.13)
            axis.axvline(start, color='#D55E00', linewidth=1.2, linestyle='--')
            axis.axvline(end, color='#D55E00', linewidth=1.2, linestyle='--')
            axis.set_ylabel(group_name)
            axis.legend(loc='upper right', ncol=len(columns), frameon=False)
            axis.grid(True, alpha=0.25)
        axes[-1].set_xlabel('Elapsed time (seconds)')
        fig.suptitle('Wrist-motion channels and trim boundaries', fontsize=14)
        plt.show()
        plt.close(fig)

def export_trimmed(_):
    if current_df is None:
        return
    trimmed = _trimmed_frame()
    start, end = time_range.value
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    action = 'kept' if trim_mode.value == 'keep' else 'removed'
    output_path = OUTPUT_DIR / (
        f'{current_path.stem}_{action}_{start:.2f}-{end:.2f}s.csv'
    )
    trimmed.to_csv(output_path, index=False)
    with status_output:
        clear_output(wait=True)
        display(HTML(
            f'<span style="color:#18794e"><b>Saved {len(trimmed):,} rows</b></span> to '
            f'<code>{output_path}</code>. The source CSV was not changed.'
        ))

load_button.on_click(load_recording)
group_picker.observe(plot_selection, names='value')
trim_mode.observe(plot_selection, names='value')
time_range.observe(plot_selection, names='value')
export_button.on_click(export_trimmed)

controls = widgets.VBox([
    widgets.HBox([file_picker, load_button]),
    widgets.HBox([group_picker, trim_mode]),
    time_range,
    widgets.HBox([reset_time, export_button]),
    status_output,
])
display(controls, plot_output)
with status_output:
    display(HTML('Choose a recording, then click <b>Load recording</b>.'))

Output()

ValueError: Missing required columns: wristMotion_rotationRateX, wristMotion_rotationRateY, wristMotion_rotationRateZ, wristMotion_gravityX, wristMotion_gravityY, wristMotion_gravityZ, wristMotion_accelerationX, wristMotion_accelerationY, wristMotion_accelerationZ, wristMotion_quaternionW, wristMotion_quaternionX, wristMotion_quaternionY, wristMotion_quaternionZ

## Optional: inspect the trimmed rows before export

Run the next cell after adjusting the controls. It shows the current output shape and a small preview.

In [ ]:
trimmed_preview = _trimmed_frame()
print(f'Output shape: {trimmed_preview.shape[0]:,} rows × {trimmed_preview.shape[1]} columns')
display(trimmed_preview.head(10))